In [1]:
! pip install torch
! pip install transformers
! pip install re

  Using cached torch-2.9.1-cp313-none-macosx_11_0_arm64.whl.metadata (30 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached torch-2.9.1-cp313-none-macosx_11_0_arm64.whl (74.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 45.2 MB/s  0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl (12 kB)
Using cached setuptools-80.9.0-py3-none-any.whl (1.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [torch]32m6/7 [torch]kx]s]
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using c

In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import re

/Users/ismail/MAI-THWS/NLP-portfolios/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

# ============================================================
# Text Cleaning Helpers
# ============================================================
def clean_text(text: str) -> str:
    # Remove ( ... ) and content inside
    text = re.sub(r"\([^)]*\)", "", text)

    # Remove [ ... ] and content inside
    text = re.sub(r"\[[^\]]*\]", "", text)

    # Remove duplicate lines
    lines = text.split("\n")
    seen = set()
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped and stripped not in seen:
            cleaned_lines.append(stripped)
            seen.add(stripped)

    # Rejoin cleaned lines
    text = "\n".join(cleaned_lines)

    return text.strip()


# ============================================================
# Load fine-tuned poetry model
# ============================================================
MODEL_PATH = "poetry_model"

print("=> Loading fine-tuned poetry model...")
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)
model = GPT2LMHeadModel.from_pretrained(MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("=> Model loaded successfully!")


# ============================================================
# Generate poem from 3 prompt words
# ============================================================
def generate_poem(prompt_words, max_length=520, temperature=0.9, top_p=0.92):
    """
    Generate a poem given three prompt words.
    """
    prompt = f"Write a poem about: {prompt_words}\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove the prompt prefix
    poem = generated_text[len(prompt) :].strip()

    # >>> NEW: clean the poem <<<
    poem = clean_text(poem)

    return poem


# ============================================================
# CLI interface
# ============================================================
if __name__ == "__main__":
    print("\n🎤 Welcome to the Poetry Slam Generator!")
    print("Enter three words to inspire your poem (e.g. 'rain love memory').")
    print("Type 'exit' to quit.\n")

    while True:
        user_input = input("🪶 Enter 3 words: ").strip()

        if user_input.lower() == "exit":
            print("👋 Goodbye!")
            break

        if len(user_input.split()) < 3:
            print("⚠️ Please enter at least three words.")
            continue

        print("\n✨ Generating your poem...\n")
        poem = generate_poem(user_input)
        print(poem)
        print("\n" + "=" * 60 + "\n")


=> Loading fine-tuned poetry model...
=> Model loaded successfully!

🎤 Welcome to the Poetry Slam Generator!
Enter three words to inspire your poem (e.g. 'rain love memory').
Type 'exit' to quit.

⚠️ Please enter at least three words.

✨ Generating your poem...

Wake up in the morning and all my troubles are gone, oh yeah.
You don't have to worry cause you can just keep on swimming, and don't you worry 'bout tomorrow.
Your hair is long, your cheeks are red, baby I love to see them grow.
If you had the strength to stand another day,
And see that we were made whole again, and I never would have let you down.
We'll find a way to make it through the hardest part of your life,
Because you are my King and I am the Queen of Rock and Roll.
We will have a perfect world but what if you don? Oh yeah, yeah
You can swim with me anywhere you like and if you want to,
I'm gonna make you mine!
And if you're lonely I'll be waiting by your side, oh yeah.
If the sun goes down you can do anything you want 